# HolidayCheck Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/holidaycheck_reviews.py`.

Covers:
1. Web scraping – fetching reviews from HolidayCheck hotel pages
2. Ollama classification – topic & sentiment detection
3. Manual review input – adding reviews that the scraper can’t fetch

**Requirements:** Ollama running locally with `mistral:7b`.
No API key needed – HolidayCheck reviews are scraped from the public website.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [2]:
from src.sites import holidaycheck_reviews as hcr
import requests

hotel_url = hcr.HC_URLS.get(hcr.ANANEA_HOTEL)
reviews_url = hcr._hotel_url_to_reviews_url(hotel_url)
print(f'Hotel: {hcr.ANANEA_HOTEL}')
print(f'Hotel URL: {hotel_url}')
print(f'Reviews URL: {reviews_url}')

Hotel: Ananea Castelo Suites Hotel
Hotel URL: https://www.holidaycheck.de/hi/ananea-castelo-suites-algarve/069563af-47db-44a3-bdb1-3441ae3a2ac4
Reviews URL: https://www.holidaycheck.de/hr/bewertungen-ananea-castelo-suites-algarve/069563af-47db-44a3-bdb1-3441ae3a2ac4


## 1. Web Scraping – Single Page

In [3]:
# Fetch a single page of reviews from HolidayCheck
resp = requests.get(reviews_url, headers=hcr.UA_HEADERS, timeout=15)
print(f'Status: {resp.status_code}')
print(f'Content length: {len(resp.text)} chars')

page_reviews = hcr._scrape_reviews_from_html(resp.text)
print(f'Reviews found on page: {len(page_reviews)}')
print()
for r in page_reviews:
    rating = hcr._normalize_rating(r.get('rating')) or '?'
    print(f"  [{r.get('id', 'N/A')[:16]}] {rating}/6 \u2013 {r.get('title', '(no title)')[:50]} ({r.get('travel_date', '')})")

Status: 200
Content length: 456627 chars
Reviews found on page: 9

  [] 6.0/6 – Sehr gerne nochmals!!! Nur zum empfehle!!!! (2025-08-27)
  [] 6.0/6 – Modernes neues Hotel mit traumhaftem Rooftop-Pool (2026-02-23)
  [] 6.0/6 – Tolles Hotel, wir waren sehr zufrieden (2025-10-12)
  [] 4.0/6 – Neues Hotel mit Potential nach Oben (2025-10-30)
  [] 6.0/6 – Erholsame Urlaubstage in einem neuen schönen Hotel (2025-10-30)
  [] 6.0/6 – insgesamt empfehlenswert (2025-11-23)
  [] 6.0/6 – Neues stilvolles Hotel (2025-11-12)
  [] 5.0/6 – Das moderne Hotel an der Algarve (2025-09-11)
  [] 6.0/6 – Architektur, Stil & unvergessliches Ambiente – ein (2025-10-06)


## 2. Web Scraping – All Pages with Pagination

In [4]:
# Fetch reviews across multiple pages
all_reviews = hcr.hc_get_reviews(hotel_url, max_pages=5, min_delay=1.0, max_delay=2.0)
print(f'Total unique reviews fetched: {len(all_reviews)}')
print()
for r in all_reviews:
    rating = hcr._normalize_rating(r.get('rating')) or '?'
    print(f"  [{r.get('id', 'N/A')[:16]}] {rating}/6 \u2013 {r.get('title', '(no title)')[:50]} ({r.get('travel_date', '')})")

Total unique reviews fetched: 19

  [e1276fbcf51afea1] 6.0/6 – Sehr gerne nochmals!!! Nur zum empfehle!!!! (2025-08-27)
  [5c353f2cb8bc3b57] 6.0/6 – Modernes neues Hotel mit traumhaftem Rooftop-Pool (2026-02-23)
  [e5943e5b40bd104e] 6.0/6 – Tolles Hotel, wir waren sehr zufrieden (2025-10-12)
  [57455503d9dd158d] 4.0/6 – Neues Hotel mit Potential nach Oben (2025-10-30)
  [333c5985699f6e1d] 6.0/6 – Erholsame Urlaubstage in einem neuen schönen Hotel (2025-10-30)
  [eeab75500d11bb6d] 6.0/6 – insgesamt empfehlenswert (2025-11-23)
  [hotel-review-32a] ?/6 –  ()
  [706c6418544c5042] 6.0/6 – Neues stilvolles Hotel (2025-11-12)
  [71d67c6f4e85a52a] 5.0/6 – Das moderne Hotel an der Algarve (2025-09-11)
  [599b4970b1af8962] 6.0/6 – Architektur, Stil & unvergessliches Ambiente – ein (2025-10-06)
  [48943a6faa27ad3b] 6.0/6 – Empfehlenswerte Adresse mit Top-Potential (2025-09-20)
  [fdab2c5f02e55296] 5.0/6 – Schönes neues ruhiges Hotel (2025-09-17)
  [32c85903307849c3] 6.0/6 – Erholung pur im Ananea

In [5]:
# Inspect raw scraped data for first review
if all_reviews:
    print(json.dumps(all_reviews[0], indent=2, ensure_ascii=False))

{
  "id": "e1276fbcf51afea1",
  "rating": 10.0,
  "title": "Sehr gerne nochmals!!! Nur zum empfehle!!!!",
  "text": "Wir waren begeistert von unserem Aufenthalt im Castelo Suites Algarve. Es war genau so, wie wir es uns vorgestellt hatten. Schon beim ersten Sprung in den Rooftop-Pool war der Alltagsstress vergessen und wir schlürften entspannt unsere Cocktails. Wir kommen höchstwahrscheinlich im Herbst nochmals! Es isch erholig puur!",
  "travel_date": "2025-08-27",
  "author_name": "Tim"
}


## 3. Ollama Classification – Test

In [6]:
# Check Ollama availability
from src.classification import classify_review, is_ollama_available

ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

Ollama available: True
Models: ['mistral:7b']


In [7]:
# Classify a single review (pick the first with text)
test_review = next((r for r in all_reviews if r.get('text')), None)
if test_review and ollama_ok:
    print(f"Review: {test_review.get('title', '(no title)')}")
    print(f"Text: {test_review['text'][:300]}...")
    print()
    topics = classify_review(test_review['text'])
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('No review with text found or Ollama not available')

Review: Sehr gerne nochmals!!! Nur zum empfehle!!!!
Text: Wir waren begeistert von unserem Aufenthalt im Castelo Suites Algarve. Es war genau so, wie wir es uns vorgestellt hatten. Schon beim ersten Sprung in den Rooftop-Pool war der Alltagsstress vergessen und wir schlürften entspannt unsere Cocktails. Wir kommen höchstwahrscheinlich im Herbst nochmals! E...

Classification result (1 topics):
  🟢 return = positive


In [8]:
# Classify ALL fetched reviews, save to JSON
from datetime import datetime

if ollama_ok:
    json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
    existing = hcr.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in all_reviews:
        text = r.get('text', '')
        if not text:
            continue
        topics = classify_review(text)

        review_id = str(r.get('id', ''))

        record = {
            'id': review_id,
            'hotel': hcr.ANANEA_HOTEL,
            'source': 'holidaycheck',
            'rating': hcr._normalize_rating(r.get('rating')),
            'title': r.get('title', ''),
            'text': text,
            'published_date': r.get('travel_date', ''),
            'author_name': r.get('author_name', ''),
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if review_id in existing_by_id else 'added'
        existing_by_id[review_id] = record

        rating = hcr._normalize_rating(r.get('rating')) or 0
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{rating}/6 [{action}] {r.get('title', 'N/A')[:45]:45s} \u2192 {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    hcr.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available \u2013 skip classification')

6.0/6 [added] Sehr gerne nochmals!!! Nur zum empfehle!!!!   → return(pos)
6.0/6 [added] Modernes neues Hotel mit traumhaftem Rooftop- → commodities(pos), comfort(pos), commodities(neg)
6.0/6 [added] Tolles Hotel, wir waren sehr zufrieden        → employees(pos), commodities(pos), return(pos)
4.0/6 [added] Neues Hotel mit Potential nach Oben           → commodities(neg), employees(pos), cleaning(pos)
6.0/6 [added] Erholsame Urlaubstage in einem neuen schönen  → commodities(pos), employees(pos), meals(pos)
6.0/6 [added] insgesamt empfehlenswert                      → commodities(pos), comfort(pos), employees(pos)
0/6 [added]                                               → commodities(pos), commodities(neg), comfort(pos)
6.0/6 [added] Neues stilvolles Hotel                        → commodities(pos), comfort(pos)
5.0/6 [added] Das moderne Hotel an der Algarve              → commodities(pos), commodities(neg)
6.0/6 [added] Architektur, Stil & unvergessliches Ambiente  → employees(pos), comm

In [ ]:
# Debug: see the raw Ollama response for a specific review
DEBUG_INDEX = 0

if ollama_ok and all_reviews:
    review = all_reviews[DEBUG_INDEX]
    text = review.get('text', '')
    print(f"Review: {review.get('title', '(no title)')}")
    print(f"Full text:\n{text}\n")
    print('--- Sending to Ollama ---')

    from src.classification import _parse_classification

    prompt = f"""You are a hotel review analyst. Read the review below carefully and identify ALL topics mentioned, even briefly. Pay special attention to complaints, cons, criticisms, and suggestions for improvement \u2013 these are NEGATIVE.

TOPICS (use these exact keys):
- employees: any mention of staff, service, friendliness, helpfulness, reception, concierge, team, waiters, management
- commodities: amenities, facilities, pool, gym, spa, room features, wifi, parking, fridge, toiletries, TV, air conditioning, balcony, shuttle, iron, entertainment, music
- comfort: room comfort, bed quality, noise, quiet, space, temperature, room size, mattress, pillow, decor, ambiance, construction noise, view
- cleaning: cleanliness, hygiene, tidiness, housekeeping, spotless, dirty, stains, towels changed, room serviced
- quality_price: value for money, pricing, worth, cost, overpriced, good deal, expensive, cheap, affordable, half board value
- meals: food, breakfast, restaurant, dining, bar, drinks, buffet, dinner, lunch, cuisine, menu, chef, kitchen, snacks, repetitive food, variety
- return: whether the guest would return, come back, visit again, recommend, revisit, not return, wouldn't go back

RULES:
1. You MUST check each topic one by one. Go through the review sentence by sentence.
2. A single review CAN and OFTEN DOES have both positive AND negative for the SAME topic.
3. Even brief or indirect mentions count.
4. If a topic is described positively, mark it positive. If negatively, mark it negative.
5. Complaints, cons, \"could be better\", \"didn't work well\", suggestions for improvement = NEGATIVE.
6. Output ONLY a JSON array. No explanation, no markdown.

Now analyze this review:
\\\"{text}\\\"

JSON array:"""

    payload = {'model': 'mistral:7b', 'prompt': prompt, 'stream': False,
               'options': {'temperature': 0.1, 'num_predict': 512}}
    resp = requests.post('http://localhost:11434/api/generate', json=payload, timeout=60)
    raw = resp.json().get('response', '')
    print(f'Raw Ollama response:\n{raw}')
    print()
    parsed = _parse_classification(raw)
    print(f'Parsed: {json.dumps(parsed, indent=2)}')

## 4. Manual Review Input

HolidayCheck pages may block scraping or return limited results.
Use this section to manually add reviews you found on the website.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'reviewer_name': 'Hans M.',                    # reviewer name (used for ID)
    'rating': 5.0,                                 # 0-6 (HolidayCheck scale)
    'title': 'Toller Urlaub',                      # review title
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
}
# ========================================

# Generate unique ID from name + date + title
id_seed = f"{manual_review['reviewer_name']}_{manual_review['published_date']}_{manual_review['title']}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

record = {
    'id': review_id,
    'hotel': hcr.ANANEA_HOTEL,
    'source': 'holidaycheck',
    'rating': manual_review['rating'],
    'title': manual_review['title'],
    'text': manual_review['text'],
    'published_date': manual_review['published_date'],
    'author_name': manual_review['reviewer_name'],
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = classify_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text \u2013 skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
existing = hcr.load_reviews(json_path)

existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'\u26a0\ufe0f  Review {record["id"]} already exists \u2013 skipping')
else:
    existing.append(record)
    hcr.save_reviews(existing, json_path)
    print(f'\u2705 Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once.

In [ ]:
import hashlib

batch_reviews = [
    {
        'reviewer_name': 'Maria S.',
        'rating': 4.5,
        'title': 'Sch\u00f6nes Hotel, kalter Pool',
        'text': 'Replace with actual review text...',
        'published_date': '2026-02-15',
    },
    # Add more reviews here...
]

json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
existing = hcr.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['reviewer_name']}_{mr['published_date']}_{mr['title']}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

    if rid in existing_ids:
        print(f'\u26a0\ufe0f  Skip duplicate: {mr["title"]}')
        continue

    rec = {
        'id': rid,
        'hotel': hcr.ANANEA_HOTEL,
        'source': 'holidaycheck',
        'rating': mr['rating'],
        'title': mr['title'],
        'text': mr['text'],
        'published_date': mr.get('published_date', ''),
        'author_name': mr['reviewer_name'],
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
    }

    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = classify_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')

    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"\u2705 {mr['title']} \u2192 {topic_str or '(unclassified)'}")

if added:
    hcr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [ ]:
json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
reviews = hcr.load_reviews(json_path)

needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            print(f"  \u2705 {r.get('title', 'N/A')[:40]} \u2192 {topic_str or '(still none)'}")
        except Exception as e:
            print(f"  \u274c {r.get('title', 'N/A')[:40]} \u2192 Error: {e}")

    hcr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

## 7. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [ ]:
json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
reviews = hcr.load_reviews(json_path)

with_text = [r for r in reviews if r.get('text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_review(r['text'])
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '\U0001f504' if old_str != new_str else '\u2705'
            print(f"  {changed} [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            print(f"  \u274c [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]} \u2192 Error: {e}")

    hcr.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

## 8. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
reviews = hcr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if "manual" in str(r.get("id", "")))}')
print()

# Rating distribution (0-6 scale)
ratings = [r.get('rating') for r in reviews if r.get('rating') is not None]
if ratings:
    print(f'Average rating: {sum(ratings)/len(ratings):.1f}/6')
    print(f'Rating range: {min(ratings):.1f} - {max(ratings):.1f}')
    print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')